In [8]:
import os, struct, math, re, json, glob
from collections import defaultdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec

In [9]:
# READ TFEVENTS

MARKER = bytes.fromhex("420808012a04")   # protobuf marker ก่อน float value

def _read_raw_events(path: str) -> List[bytes]:
    """อ่าน raw bytes ของทุก event จาก tfevents file"""
    events = []
    with open(path, "rb") as f:
        while True:
            header = f.read(8)
            if len(header) < 8:
                break
            n = struct.unpack("Q", header)[0]
            f.read(4)                       # CRC ของ header
            data = f.read(n)
            if len(data) < n:
                break
            f.read(4)                       # CRC ของ data
            events.append(data)
    return events


def _get_step(ev: bytes) -> Optional[int]:
    """ดึง global_step จาก event bytes"""
    i = 0
    while i < len(ev) - 1:
        b = ev[i]; i += 1
        wire = b & 7; field = b >> 3
        if wire == 0:
            val = 0; sh = 0
            while i < len(ev):
                b2 = ev[i]; i += 1
                val |= (b2 & 0x7F) << sh; sh += 7
                if not (b2 & 0x80): break
            if field == 2:
                return val
        elif wire == 1:
            i += 8
        elif wire == 2:
            ln = 0; sh = 0
            while i < len(ev):
                b2 = ev[i]; i += 1
                ln |= (b2 & 0x7F) << sh; sh += 7
                if not (b2 & 0x80): break
            i += ln
        elif wire == 5:
            i += 4
        else:
            break
    return None


def _parse_scalars(ev: bytes) -> List[Tuple[str, float]]:
    """ดึงคู่ (tag, value) ทั้งหมดจาก event"""
    results = []
    pos = 0
    while pos < len(ev):
        idx = ev.find(MARKER, pos)
        if idx == -1:
            break
        fp = idx + 6
        if fp + 4 > len(ev):
            break
        value = struct.unpack("<f", ev[fp : fp + 4])[0]
        tag = None
        seg = ev[max(0, idx - 80) : idx]
        for j in range(len(seg) - 2, -1, -1):
            if seg[j] == 0x0A:
                sl = seg[j + 1]
                if 2 < sl < 80 and j + 2 + sl <= len(seg):
                    cand = seg[j + 2 : j + 2 + sl]
                    try:
                        s = cand.decode("utf-8")
                        if all(32 <= ord(c) < 127 for c in s):
                            tag = s
                            break
                    except Exception:
                        pass
        if tag and not math.isnan(value):
            results.append((tag, value))
        pos = idx + 1
    return results


def load_tfevents(path: str) -> Dict[str, Dict]:
    """
    อ่าน tfevents file และคืนค่า dict:
        { tag: {"steps": [...], "values": [...]} }
    """
    raw = _read_raw_events(path)
    bucket = defaultdict(list)
    for ev in raw:
        step = _get_step(ev)
        for tag, val in _parse_scalars(ev):
            if step is not None:
                bucket[tag].append((step, val))
    result = {}
    for tag, pairs in bucket.items():
        pairs.sort(key=lambda x: x[0])
        result[tag] = {
            "steps":  [p[0] for p in pairs],
            "values": [p[1] for p in pairs],
        }
    return result


def smooth(values: List[float], window: int = 7) -> List[float]:
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="same").tolist()


# ─── key tags ──────────────────────────────────────────────
_TAGS = {
    "main_loss":      "Step_Stats/train/Losses/train_all_loss",
    "loss_epoch":     "Losses/train_all_loss",
    "presence_loss":  "Losses/train_all_presence_loss",
    "presence_acc":   "Losses/train_all_presence_dec_acc",
    "bbox_loss":      "Losses/train_all_loss_bbox",
    "giou_loss":      "Losses/train_all_loss_giou",
    "ce_loss":        "Losses/train_all_loss_ce",
    "ce_f1":          "Losses/train_all_ce_f1",
    "ce_f1_o2m":      "Losses/train_all_ce_f1_o2m",
    "coco_ap":        "Meters_train/val_ive/detection/coco_eval_bbox_AP",
    "coco_ap50":      "Meters_train/val_ive/detection/coco_eval_bbox_AP_50",
    "lr":             "Optim/0_/lr",
    "batch_time":     "Step_Stats/train/Batch Time",
    "mem_gb":         "Step_Stats/train/Mem (GB)",
    "progress":       "Trainer/where",
    "epoch":          "Trainer/epoch",
}


def plot_training_dashboard(
    tfevents_path: str,
    save_dir: str = ".",
    output_file: str = ".",
    dark_mode: bool = True,
) -> str:
    """
    อ่าน tfevents แล้ว plot training dashboard ครบทุก metric

    Parameters
    ----------
    tfevents_path : path ไปยัง tfevents file
    save_dir      : directory ที่จะบันทึก PNG
    dark_mode     : ใช้ dark theme หรือไม่

    Returns
    -------
    path ของไฟล์ที่บันทึก
    """
    logs = load_tfevents(tfevents_path)

    def g(key):
        tag = _TAGS.get(key, key)
        return logs.get(tag, {"steps": [], "values": []})

    # ─── theme ──────────────────────────────────────────────
    BG   = "#0f1117" if dark_mode else "#f8fafc"
    CARD = "#1a1d2e" if dark_mode else "#ffffff"
    TXT  = "#e2e8f0" if dark_mode else "#1e293b"
    MUT  = "#6b7280" if dark_mode else "#94a3b8"
    BLUE, GREEN, ORG, RED, YEL = "#4f8ef7", "#3ecf8e", "#f97316", "#ef4444", "#facc15"
    GRID = "#2d3148" if dark_mode else "#e2e8f0"

    def style(ax, title):
        ax.set_facecolor(CARD)
        for sp in ax.spines.values():
            sp.set_color(GRID)
        ax.tick_params(colors=MUT, labelsize=8)
        ax.xaxis.label.set_color(MUT)
        ax.yaxis.label.set_color(MUT)
        ax.set_title(title, color=TXT, fontsize=10, fontweight="bold", pad=8)
        ax.grid(True, color=GRID, linewidth=0.5, alpha=0.8)

    fig = plt.figure(figsize=(18, 12))
    fig.patch.set_facecolor(BG)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # 1. Main loss (step-level, log scale)
    ax = fig.add_subplot(gs[0, :2])
    style(ax, "Training Loss  (step-level, log scale)")
    d = g("main_loss")
    if d["steps"]:
        sv = smooth(d["values"], 7)
        ax.plot(d["steps"], d["values"], color=BLUE, alpha=0.2, lw=0.8)
        ax.plot(d["steps"], sv, color=BLUE, lw=2, label="train_all_loss (smoothed)")
        ax.set_yscale("log")
        ax.set_xlabel("Step")
        ax.legend(fontsize=8, labelcolor=TXT, facecolor=CARD, edgecolor=MUT)

    # 2. Learning rate
    ax = fig.add_subplot(gs[0, 2])
    style(ax, "Learning Rate  (Optim/0_)")
    d = g("lr")
    if d["steps"]:
        ax.plot(d["steps"], d["values"], color=YEL, lw=1.8)
        ax.set_xlabel("Step")

    # 3. Val Detection AP
    ax = fig.add_subplot(gs[1, 0])
    style(ax, "Val Detection AP")
    for key, col, lbl in [("coco_ap", GREEN, "AP"), ("coco_ap50", BLUE, "AP@50")]:
        d = g(key)
        if d["steps"]:
            ax.plot(d["steps"], d["values"], color=col, lw=2,
                    marker="o", ms=5, label=lbl)
    ax.set_ylim(0, 1); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8, labelcolor=TXT, facecolor=CARD, edgecolor=MUT)

    # 4. Loss breakdown per epoch
    ax = fig.add_subplot(gs[1, 1])
    style(ax, "Loss Breakdown  (per epoch)")
    for key, col, lbl in [
        ("bbox_loss", ORG, "bbox"),
        ("giou_loss", YEL, "giou"),
        ("ce_loss",   BLUE, "ce"),
        ("presence_loss", RED, "presence"),
    ]:
        d = g(key)
        if d["steps"]:
            ax.plot(d["steps"], d["values"], color=col, lw=1.5, label=lbl)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=7, labelcolor=TXT, facecolor=CARD, edgecolor=MUT)

    # 5. Presence decoder accuracy
    ax = fig.add_subplot(gs[1, 2])
    style(ax, "Presence Decoder Accuracy")
    d = g("presence_acc")
    if d["steps"]:
        ax.plot(d["steps"], d["values"], color=GREEN, lw=2)
        ax.axhline(1.0, color=MUT, ls="--", lw=0.8, alpha=0.5)
        ax.set_ylim(0.7, 1.05)
    ax.set_xlabel("Epoch")

    # 6. CE F1
    ax = fig.add_subplot(gs[2, 0])
    style(ax, "CE F1  (per epoch)")
    for key, col, lbl in [("ce_f1", BLUE, "ce_f1"), ("ce_f1_o2m", ORG, "ce_f1_o2m")]:
        d = g(key)
        if d["steps"]:
            ax.plot(d["steps"], d["values"], color=col, lw=2, label=lbl)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8, labelcolor=TXT, facecolor=CARD, edgecolor=MUT)

    # 7. Batch time (filtered)
    ax = fig.add_subplot(gs[2, 1])
    style(ax, "Batch Time  (sec, warmup filtered)")
    d = g("batch_time")
    if d["steps"]:
        pairs = [(s, v) for s, v in zip(d["steps"], d["values"]) if v < 10]
        if pairs:
            ss, vv = zip(*pairs)
            ax.plot(ss, vv, color=ORG, lw=1.5)
    ax.set_xlabel("Step")

    # 8. Training progress
    ax = fig.add_subplot(gs[2, 2])
    style(ax, "Training Progress (%)")
    d = g("progress")
    if d["steps"]:
        pct = [v * 100 for v in d["values"]]
        ax.fill_between(d["steps"], pct, alpha=0.3, color=GREEN)
        ax.plot(d["steps"], pct, color=GREEN, lw=2)
        ax.set_ylim(0, 105); ax.set_xlabel("Epoch")
        ax.axhline(pct[-1], color=YEL, ls="--", lw=1, alpha=0.7)
        ax.text(d["steps"][-1], pct[-1] + 2, f"{pct[-1]:.1f}%", color=YEL, fontsize=9)

    # summary stats
    def last_val(key):
        d = g(key)
        return d["values"][-1] if d["values"] else float("nan")
    def best_val(key, mode="min"):
        d = g(key)
        if not d["values"]: return float("nan"), float("nan")
        fn = min if mode == "min" else max
        v = fn(d["values"]); s = d["steps"][d["values"].index(v)]
        return v, s

    best_loss, best_loss_step = best_val("main_loss", "min")
    best_ap,   best_ap_epoch  = best_val("coco_ap",   "max")

    title = (
        f"SAM3 Fine-tune Dashboard  |  "
        f"Epochs: {int(last_val('epoch'))}  |  "
        f"Best Loss: {best_loss:.3f} (step {int(best_loss_step):,})  |  "
        f"Best AP: {best_ap:.4f} (epoch {int(best_ap_epoch)})"
    )
    fig.suptitle(title, color=TXT, fontsize=12, fontweight="bold", y=0.995)

    os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, output_file)
    plt.savefig(out, dpi=150, bbox_inches="tight",
                facecolor=BG, edgecolor="none")
    plt.close()
    print(f"[dashboard] saved -> {out}")
    return out

In [10]:
TFEVENTS_PATH = "outputs/ive_training/tensorboard/events.out.tfevents.1771856849.ba7ecb4f82af.929.079dc5e2e-e3bc-4ba9-aa7c-377b2d8a3a1c"

plot_training_dashboard(
    tfevents_path = TFEVENTS_PATH,
    save_dir      = "outputs/train_logs",
    output_file   = "training_dashboard_dark.png",
    dark_mode     = True,
)

[dashboard] saved -> outputs/train_logs\training_dashboard_dark.png


'outputs/train_logs\\training_dashboard_dark.png'

In [11]:
plot_training_dashboard(
    tfevents_path = TFEVENTS_PATH,
    save_dir      = "outputs/train_logs",
    output_file   = "training_dashboard_light.png",
    dark_mode     = False,
)

[dashboard] saved -> outputs/train_logs\training_dashboard_light.png


'outputs/train_logs\\training_dashboard_light.png'